# Customizing Agent Memory

In [3]:
from langchain.agents import create_agent 
from langchain.agents.middleware import SummarizationMiddleware 
from langgraph.checkpoint.memory import InMemorySaver
from langchain_mistralai import ChatMistralAI 
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
model = ChatMistralAI(
    model = "mistral-medium-2508"
)

In [ ]:
agent = create_agent(
    model = model,
    middleware=[SummarizationMiddleware(
        model=model,
        trigger=("tokens",1000),
        keep = ("messages",2)
    )],
    checkpointer=InMemorySaver()
)

In [16]:
config = {"configurable":{"thread_id":"1"}}

for i in range(5):
    user_input = input("user : ")
    response = agent.invoke(
        {
            "messages" : [("user",user_input)]
        },
        config = config
    )
    print(f"Ai response : {response['messages'][-1].content}")

Ai response : Hello, Dhairya! 😊 It's so nice to meet you! Blue is an *amazing* color—it’s calming, deep, and full of meaning (like the sky, ocean, or even trust and wisdom). Do you have a favorite shade of blue? Maybe navy, turquoise, or electric blue?

Also, fun fact: Blue is the most *universally loved* color in the world! 🌍💙 What do you love most about it? (Is it the vibe, the aesthetics, or something else?)

P.S. Your name "Dhairya" is super cool too—it means *patience* and *courage* in Sanskrit, which is such a powerful combo! ✨
Ai response : That’s a fantastic combo—**India** and **Rohit Sharma**—two things that make every Indian proud! Let’s break it down:

---

### **🇮🇳 About India: The Land of Diversity & Wonders**
India (*Bharat*) is a **7,000-year-old civilization** with a mix of ancient traditions and modern dynamism. Here’s why it’s incredible:

1. **Geography & Culture**
   - **7th largest country** by area, with **28 states & 8 UTs** (from the Himalayas to tropical Keral

In [18]:
response

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n```\n## SESSION INTENT\nThe user (Dhairya) requested detailed information about **Rohit Sharma’s luxury car collection**, including models, estimated values, and notable features, after initially exploring his cricket career.\n\n## SUMMARY\n- **Primary Focus**: Rohit Sharma’s **8 confirmed/rumored cars**, their specifications, and fun facts.\n  - **Key Cars**:\n    - **Lamborghini Urus** (₹4.2 Crore+, gifted by wife Ritika Sajdeh).\n    - **Mercedes-Benz GLE 450 AMG** (₹1.5 Crore+, family SUV).\n    - **BMW 7 Series** (₹1.5 Crore+, official events).\n    - **Porsche 911** (₹1.8 Crore+, rumored sports car).\n    - **Range Rover Vogue** (₹2.5 Crore+, long drives).\n    - **Total collection worth**: **~₹12–15 Crores** ($1.5–2M+).\n  - **Fun Facts**:\n    - First car: **Maruti Suzuki Esteem** (gift from his father).\n    - Wife’s influence: Ritika Sajdeh helped pick the **Lamborghini Urus** (30th birthday

# Prompt - manipulation 

In [23]:
from langchain.agents import create_agent
from typing import TypedDict
from langchain.agents.middleware import dynamic_prompt, ModelRequest,before_model,AgentMiddleware,AgentState
from langgraph.runtime import Runtime

In [37]:
class CustomContext(TypedDict):
    user_name : str 

class WelcomeMiddleWare(AgentMiddleware):

    def before_model(self,state:AgentState,runtime:Runtime) -> None:
        name = runtime.context['user_name'] 
        print(f"welcome {name} , how can I help you.")
        return None

@dynamic_prompt 
def generate_dynamic_prompt(request : ModelRequest) -> str:
    user_name = request.runtime.context['user_name']
    system_prompt = f"You are a helpful assistant. address the user as {user_name}"
    return system_prompt 


In [53]:
agent = create_agent(
    model = "mistral-small-latest",
    middleware=[WelcomeMiddleWare(),generate_dynamic_prompt],
    context_schema = CustomContext
)


In [54]:
response = agent.invoke(
    {
        "messages" : [('user',"hello do you know my name")]
    },
    context = CustomContext(user_name="Dhairya")
)

welcome Dhairya , how can I help you.


In [55]:
response 

{'messages': [HumanMessage(content='hello do you know my name', additional_kwargs={}, response_metadata={}, id='0a85e291-363f-41b1-9ed6-78fdfbf31b11'),
  AIMessage(content="Hello Dhairya! Yes, I can use the name you've provided to address you. How can I assist you today?", additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 24, 'total_tokens': 51, 'completion_tokens': 27, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-latest', 'model': 'mistral-small-latest', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--019cdcea-6c2f-73d2-a521-a7ce7ebab426-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 24, 'output_tokens': 27, 'total_tokens': 51})]}

# After Model

In [56]:
from langchain.messages import RemoveMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import after_model
from langgraph.runtime import Runtime

In [71]:
@after_model
def Validater(state: AgentState, runtime: Runtime) -> dict | None:
    """
    if last messages contains [password,secret] words the messages will be deleted.
    """
    STOP_WORDS = ["password", "secret"]
    last_message = state["messages"][-1]
    if any(word in last_message.content for word in STOP_WORDS):
        return {"messages": [RemoveMessage(id=last_message.id)]}
    return None

agent = create_agent(
    model="mistral-small-latest",
    middleware=[Validater],
    checkpointer=InMemorySaver(),
)

In [76]:
config = {"configurable":{"thread_id":"1"}}
response = agent.invoke(
    {
        "messages" : [("user","Tell me about Moica geller")]
    },
    config = config
)



In [77]:
response

{'messages': [HumanMessage(content='tell me about Ross geller in short.', additional_kwargs={}, response_metadata={}, id='2448ca06-2f08-4419-b363-d99fb20fee3c'),
  AIMessage(content='Ross Geller is a fictional character from the popular TV show *Friends* (1994–2004). Played by David Schwimmer, Ross is a paleontologist (later a professor) known for his intelligence, awkwardness, and often comedic misfortunes. Key traits include:\n\n- **Nerdy & Insecure**: Often overthinks situations, especially in relationships.\n- **Divorce Troubles**: Married and divorced multiple times (including to Rachel and Carol).\n- **Jealousy**: Struggles with insecurities, especially around Rachel.\n- **Catchphrases**: "We were on a break!" and "PIVOT!" (from moving a couch).\n- **Fatherhood**: Becomes a dad to Ben and later Emma (with Rachel).\n\nRoss is one of the show’s central characters, balancing humor with heartfelt moments.', additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 13,